# Урок 7. Регрессия: предсказываем число

До сих пор мы предсказывали **категорию**: выжил / погиб. Это **классификация**.
Теперь предсказываем **число**: возраст в годах. Это **регрессия**.

| | Классификация | Регрессия |
|---|---|---|
| Что предсказываем | категорию (0 или 1) | число (28.5 лет) |
| Как меряем | accuracy, precision, recall | MAE, RMSE, R² |
| Базовая линия | «все погибли» | «всем одно и то же число» |

Всё остальное то же самое: `split` → `fit` → `predict` → метрика → кросс-валидация.

## 0. Подготовка

Возраст известен не у всех: у 177 человек пропуск. Обучать можно только на тех, у кого возраст **есть** (714 человек).
Признак `Age` теперь **ответ** (`y`), а не признак. Остальные колонки (пол, класс, титул) — признаки.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/titanic.csv")
df["Title"] = (df["Name"].str.split(", ").str[1].str.split(" ").str[0].str.rstrip("."))
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

known = df[df["Age"].notna()].copy()     # только те, у кого возраст известен
y = known["Age"]                         # ответ = возраст (число!)

num_cols = ["Pclass", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked", "Title"]

def make_preprocess():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore")), cat_cols),
    ])

# train_test_split без stratify: для чисел группы по классам не нужны
X_train, X_test, y_train, y_test = train_test_split(known, y, test_size=0.2, random_state=42)
print(len(known), len(X_train), len(X_test))
print(f"средний возраст {y.mean():.1f}, медиана {y.median():.1f}, разброс (std) {y.std():.1f}")

714 571 143
средний возраст 29.7, медиана 28.0, разброс (std) 14.5


## 1. Метрики регрессии

Допустим, модель предсказала возраст 3 людям:

| Реально | Предсказано | Ошибка | |Ошибка| | Ошибка² |
|---|---|---|---|---|
| 20 | 25 | +5 | 5 | 25 |
| 30 | 28 | −2 | 2 | 4 |
| 40 | 50 | +10 | 10 | 100 |

- **MAE** (средняя абсолютная ошибка) = (5 + 2 + 10) / 3 = **5.7 года**. Читается просто: «в среднем ошибается на 5.7 лет».
- **RMSE** = корень из среднего квадрата ошибок = √((25 + 4 + 100) / 3) = **6.6 года**. Большие ошибки (как 10) штрафуются сильнее. Поэтому RMSE ≥ MAE всегда.
- **R²**: 0 = модель не лучше, чем «всем среднее»; 1 = идеально; **меньше 0 = хуже, чем «всем среднее»**.

In [2]:
def report(name, y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = mean_squared_error(y_true, pred) ** 0.5
    r2 = r2_score(y_true, pred)
    print(f"{name:10s} MAE = {mae:5.2f} лет   RMSE = {rmse:5.2f}   R² = {r2:6.3f}")

# проверка на примере из таблицы
report("пример", [20, 30, 40], [25, 28, 50])

пример     MAE =  5.67 лет   RMSE =  6.56   R² =  0.355


## 2. Базовая линия: всем одно число

Как «все погибли» в классификации. Модель с `strategy="median"` всегда отвечает медианой возраста из учебной части.

In [3]:
base = Pipeline([("prep", make_preprocess()),
                 ("model", DummyRegressor(strategy="median"))])
base.fit(X_train, y_train)
print("модель всем говорит:", base[-1].constant_.item(), "лет")
report("медиана", y_test, base.predict(X_test))

модель всем говорит: 28.5 лет
медиана    MAE = 10.80 лет   RMSE = 13.62   R² = -0.000


**Вывод:** если всем говорить 28.5 лет, ошибка в среднем **10.8 лет**, R² = 0. Это число, которое надо побить.
Разброс возраста (std) 14.5, а MAE 10.8. Это нормально: MAE обычно меньше std.

## 3. Линейная регрессия

То же, что логистическая, но ответ не вероятность, а число. Модель считает: **возраст = 28.2 + баллы признаков**.

In [4]:
lin = Pipeline([("prep", make_preprocess()), ("model", LinearRegression())])
lin.fit(X_train, y_train)
report("линейная", y_test, lin.predict(X_test))

names = [n.split("__", 1)[1] for n in lin[0].get_feature_names_out()]
pd.Series(lin[-1].coef_, index=names).round(1).sort_values()

линейная   MAE =  8.30 лет   RMSE = 10.25   R² =  0.434


Title_Master   -21.8
Pclass          -5.6
Sex_female      -3.8
Embarked_C      -3.2
SibSp           -1.9
Title_Miss      -1.8
Embarked_S      -1.4
Fare            -0.4
Parch           -0.2
Title_Mr         2.6
Sex_male         3.8
Embarked_Q       4.6
Title_Rare       8.9
Title_Mrs       12.0

**Вывод:** MAE упал с 10.8 до **8.3** года, R² = 0.43. Модель объясняет заметную часть разброса возраста.

**Как читать коэффициенты** (это «годы», а не «баллы»):
- `Title_Master` = −21.8: мальчик (Master) на 22 года **моложе** среднего.
- `Title_Mrs` = +12.0: замужняя женщина на 12 лет **старше**.
- `Pclass` = −5.6: чем больше номер класса (3-й), тем моложе. У числовых признаков масштаб «на 1 std», как в уроке 4.
- `Sex_male` +3.8 и `Sex_female` −3.8 — зеркальные. Это особенность one-hot. Смотри на **разницу** между ними (7.6 года), а не на каждое по отдельности.

## 4. Случайный лес для регрессии

Тот же лес, но в листе вместо большинства берётся **среднее** возраста.

In [5]:
forest = Pipeline([("prep", make_preprocess()),
                   ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                   random_state=42, n_jobs=-1))])
forest.fit(X_train, y_train)
report("лес", y_test, forest.predict(X_test))

лес        MAE =  7.78 лет   RMSE =  9.87   R² =  0.475


**Вывод:** лес чуть лучше линейной: MAE 7.78 против 8.30. Но тест из 143 человек шумный, одно сравнение ничего не доказывает. Проверим кросс-валидацией.

## 5. Кросс-валидация и ловушка «минус»

В `sklearn` для всех метрик правило: **больше = лучше**. Но ошибка должна быть **меньше**. Поэтому для MAE берём `neg_mean_absolute_error`: это MAE со знаком минус. Чем ближе к нулю, тем лучше.
Чтобы получить обычный MAE, умножаем на −1.

Для регрессии используем `KFold`, а не `StratifiedKFold`: «стратифицировать» число нельзя.

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in [("медиана", DummyRegressor(strategy="median")),
                    ("линейная", LinearRegression()),
                    ("лес", RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                  random_state=42, n_jobs=-1))]:
    pipe = Pipeline([("prep", make_preprocess()), ("model", model)])
    r = cross_validate(pipe, X_train, y_train, cv=kf,
                       scoring=["neg_mean_absolute_error", "r2"])
    rows.append({"модель": name,
                 "MAE": -r["test_neg_mean_absolute_error"].mean(),   # минус убираем
                 "std MAE": r["test_neg_mean_absolute_error"].std(),
                 "R²": r["test_r2"].mean()})

pd.DataFrame(rows).round(3)

,модель,MAE,std MAE,R²
0,медиана,11.409,0.443,-0.018
1,линейная,9.089,0.523,0.385
2,лес,8.711,0.653,0.402


**Вывод:**
- Медиана: MAE 11.41, R² ≈ 0.
- Линейная: MAE 9.09, R² 0.385.
- Лес: MAE 8.71, R² 0.402.

Обе модели намного лучше базовой линии (на 2–3 года). Лес лучше линейной на 0.38 года. Но `std MAE` у них 0.52 и 0.65: разброс **больше** разницы. Значит, между лесом и линейной регрессией разница в пределах шума. Оба обогнали медиану уверенно: там разница 2.3–2.7 года.

## 6. На ком ошибается модель?

Как в уроке 3: средняя ошибка скрывает, **где** модель плоха. Смотрим MAE леса по титулам на тесте.

In [7]:
pred = forest.predict(X_test)
by_title = (X_test.assign(error=(y_test - pred).abs())
                  .groupby("Title")["error"].agg(["mean", "size"]))
by_title.columns = ["MAE", "людей в тесте"]
by_title.round(1)

,MAE,людей в тесте
Title,,
Master,3.3,11
Miss,6.7,31
Mr,7.9,73
Mrs,11.1,22
Rare,7.6,6


**Вывод:**
- `Master` (мальчики): MAE 3.3 года. Легко: все они маленькие (медиана 3.5 года).
- `Mrs`: MAE 11.1 года. Трудно: замужние женщины бывают и 20, и 60 лет, а признаков, которые это различают, у нас нет.
- Осторожно: в тесте всего 11 `Master`, 6 `Rare`. По таким группам выводы шаткие. Смотри на колонку `людей в тесте` (это **количество**, не доля).

**Практическая идея:** такую модель можно использовать вместо медианы, чтобы заполнять пропуски возраста (`SimpleImputer`). Для `Master` это точно лучше, чем 28 лет.

---
# 🏠 Домашка

## Задание 1. Считаем вручную

Реальный возраст: [10, 30, 50]. Модель предсказала: [15, 25, 60].
Посчитай **MAE** и **RMSE** вручную (на бумаге), потом проверь кодом через `report`.

## Задание 2. Подбор для леса (урок 6 + урок 7)

Возьми `forest` из части 4 и сделай `GridSearchCV` на `X_train`, `y_train`:
- `model__max_depth`: [3, 5, 8, None];
- `model__min_samples_leaf`: [1, 3, 5, 10, 20];
- `cv=kf`, `scoring="neg_mean_absolute_error"`, `n_jobs=-1`.

Выведи:
1. сколько всего обучений будет (посчитай до запуска);
2. `best_params_`;
3. `best_score_` как **обычный MAE** (не забудь про минус!);
4. MAE на тесте через `report`.

**Вывод словами:** лучше ли подобранный лес, чем лес из части 4 (MAE 7.78 на тесте)? Разница больше шума или нет?

## Задание 3. Вопрос без кода

а) Модель получила R² = −0.3. Что это значит простыми словами?
б) Ты говоришь начальнику: «MAE модели 8 лет». Как объяснишь, что это значит, человеку без знаний ML?

**Ответ:**

In [8]:
# задание 1: проверка (функция report определена в части 1)
report("пример", [10, 30, 50], [15, 25, 60])

пример     MAE =  6.67 лет   RMSE =  7.07   R² =  0.812


### Ответ на задание 1

Ошибки: |15 − 10| = 5, |25 − 30| = 5, |60 − 50| = 10.

- **MAE** = (5 + 5 + 10) / 3 = **6.67**.
- **RMSE** = √((25 + 25 + 100) / 3) = √50 = **7.07**.

RMSE больше MAE, потому что ошибки сначала возводят в квадрат, потом усредняют и берут корень. Большая ошибка (10) в квадрате даёт 100 и сильнее тянет среднее вверх. Поэтому RMSE сильнее реагирует на большие ошибки. RMSE ≥ MAE всегда.

In [9]:
# задание 2: GridSearchCV для леса-регрессора
forest_base = Pipeline([("prep", make_preprocess()),
                        ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                        random_state=42, n_jobs=-1))])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "model__max_depth":        [3, 5, 8, None],
    "model__min_samples_leaf": [1, 3, 5, 10, 20],
}

n_combinations = int(np.prod([len(v) for v in param_grid.values()]))
print(f"комбинаций: {n_combinations}, обучений: {n_combinations * kf.get_n_splits()}")

search_rf = GridSearchCV(forest_base, param_grid, cv=kf,
                         scoring="neg_mean_absolute_error", n_jobs=-1)
search_rf.fit(X_train, y_train)

print(search_rf.best_params_)
print(f"cv MAE:  {-search_rf.best_score_:.3f}")     # минус убираем
report("тест", y_test, search_rf.predict(X_test))

комбинаций: 20, обучений: 100
{'model__max_depth': 5, 'model__min_samples_leaf': 1}
cv MAE:  8.530
тест       MAE =  7.98 лет   RMSE = 10.04   R² =  0.456


### Ответ на задание 2

1. Комбинаций 4 × 5 = 20. **Обучений** 20 × 5 частей = **100**.
2. `best_params_`: `max_depth=5`, `min_samples_leaf=1`.
3. cv MAE = **8.530** (после умножения `best_score_` на −1).
4. Тест MAE = **7.98**.

**Сравнение** — всегда cv с cv и тест с тестом. cv и тест — разные измерения, их нельзя вычитать друг из друга. Меньше MAE = лучше.

| | Лес из части 4 | Лес после подбора |
|---|---|---|
| cv MAE | 8.71 | 8.53 (лучше на 0.18) |
| тест MAE | 7.78 | 7.98 (хуже на 0.20) |

cv говорит «лучше», тест говорит «хуже», обе разницы около 0.2, а разброс между частями (std) 0.68. **Вывод:** подбор ничего не дал, это шум. Как и в уроке 6.

**Три вида оценки:**
- **train** — на тех же людях, на которых модель училась (всегда оптимистично);
- **cv** — по частям внутри учебной части, каждый раз на людях, которых модель в этом раунде не видела;
- **test** — на отложенных 143 людях, которых не видел даже поиск.

### Ответ на задание 3

**а) R² = −0.3.** R² = 0 значит «модель не лучше, чем всем говорить среднее». R² = −0.3 значит **хуже**, чем всем говорить среднее. Модель бесполезна, или в ней ошибка (например, перепутаны данные).

**б) MAE = 8 лет.** В среднем прогноз отличается от настоящего возраста примерно на 8 лет. Это не значит, что каждая ошибка ровно 8 лет: где-то модель ошибается меньше, где-то больше. Чтобы понять, много это или мало, надо сравнить с базовой линией: «всем одно число» ошибается на 10.8 года, а модель на 8.